In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
from scipy import signal
import matplotlib.image as mpimg

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 13 &mdash; Wavelet Transforms and Their Applications</h1></center>

The spectrogram of Module 12 showed how the frequency content of a chirp, a passing train and the 2011 Tohoku earthquake changes with time,
but it forced one choice on every frequency: the window length. A long window separates closely spaced low frequencies and smears short
transients, while a short window does the opposite, and real geophysical records (a long-period surface wave arriving after a sharp P wave, a
reflection sitting on top of low-frequency ground roll) contain both at once. Wavelets solve this by using short analysis windows for high
frequencies and long ones for low frequencies, and the Ricker wavelet of the seismic convolutional model in Module 7 turns out to be one of them.
This module builds the continuous wavelet transform (CWT) and the discrete wavelet transform (DWT) from about 35 lines of NumPy, applies the CWT
to a chirp and to the Tohoku record, and uses the DWT to denoise a trace and compress an image.

# Outline
- [13.1 Motivation: One Window Does Not Fit All](#13.1-Motivation:-One-Window-Does-Not-Fit-All)
- [13.2 Tiling the Time–Frequency Plane](#13.2-Tiling-the-Time–Frequency-Plane)
- [13.3 Wavelets: Little Waves](#13.3-Wavelets:-Little-Waves)
- [13.4 The Continuous Wavelet Transform](#13.4-The-Continuous-Wavelet-Transform)
  - [13.4.1 Computing the CWT with the FFT](#13.4.1-Computing-the-CWT-with-the-FFT)
  - [13.4.2 From scale to frequency](#13.4.2-From-scale-to-frequency)
- [13.5 CWT Examples](#13.5-CWT-Examples)
- [13.6 The Discrete Wavelet Transform: the Haar Filter Bank](#13.6-The-Discrete-Wavelet-Transform:-the-Haar-Filter-Bank)
  - [13.6.1 Averages and differences](#13.6.1-Averages-and-differences)
  - [13.6.2 Downsampling by two without losing information](#13.6.2-Downsampling-by-two-without-losing-information)
  - [13.6.3 Daubechies-4: a smoother filter pair](#13.6.3-Daubechies-4:-a-smoother-filter-pair)
- [13.7 Applications: Denoising and Compression](#13.7-Applications:-Denoising-and-Compression)
  - [13.7.1 Denoising by thresholding](#13.7.1-Denoising-by-thresholding)
  - [13.7.2 Compressing a 2D image](#13.7.2-Compressing-a-2D-image)
- [13.8 Parting Thoughts](#13.8-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="13.1"></a>
## 13.1 Motivation: One Window Does Not Fit All

The purpose of this section is for you to get experience in looking at how the spectral content can change in datasets represented by long time series.  So far we have looked at how the Fourier Transform can be applied to look at the spectral content. However, the Fourier Transform is a **global** operation that cannot distinguish how the frequency content varies within a signal.  Examples of the global nature are evident when one thinks about the following Fourier transform pair:

$$\delta(t-t_0) \leftrightarrow e^{-i\omega t_0} \tag{1}$$

and 

$$\cos (\omega_0 t) + i\sin(\omega_0 t) \leftrightarrow 2\pi\,\delta(\omega-\omega_0) \tag{2}$$

In equation (1), a $\delta$-function spike at one moment in time affects everywhere in the frequency domain.  In equation (2), a $\delta$-function spike at one frequency has an effect everywhere in the time domain.  Of course, we have looked at other less extreme examples where the effects are more restricted in the time and frequency domains.  For example, the Fourier transform of Gaussian function in the time domain is a Gaussian function in the frequency domain.

Module 12 (Section 12.1) made the same argument in terms of <b>stationarity</b>: a single Fourier transform assumes that the spectral content does not evolve with time, while a passing train or an earthquake record plainly does. The spectrogram answered by cutting the record into short windows. This module asks whether the window itself can change with frequency. Figure 13.1 shows why that matters.

In [ ]:
C = DSP_COLORS
# . . Test signal: a 4 Hz + 8 Hz tone pair centered at 1 s, and two clicks 30 ms apart at 3 s
fs_test = 1000.0
t_test = np.arange(0, 4, 1/fs_test)
tones = (np.cos(2*np.pi*4*t_test) + np.cos(2*np.pi*8*t_test))*np.exp(-((t_test - 1.0)/0.6)**2)
clicks = np.zeros_like(t_test); clicks[[3000, 3030]] = 1
x_test = tones + 6*np.convolve(clicks, signal.windows.hann(5), mode='same')

def stft_power(x, nper):
    f, t, S = signal.spectrogram(x, fs_test, nperseg=nper, noverlap=nper - max(1, nper//50), nfft=4096)
    return f, t, S

f_long, t_long, S_long = stft_power(x_test, 1000)     # 1 s window
f_short, t_short, S_short = stft_power(x_test, 16)    # 16 ms window

fig, axs = plt.subplots(3, 1, figsize=(10, 7.2), gridspec_kw=dict(height_ratios=[0.8, 1, 1]))
axs[0].plot(t_test, x_test, color=C['time_domain'], lw=0.8)
axs[0].set_xlim(0, 4); axs[0].set_ylabel('amplitude'); axs[0].set_title('(a) Test signal: tone pair (4 + 8 Hz) and two clicks 30 ms apart', fontsize=12)
for ax, (f, t, S, lab) in zip(axs[1:], [(f_long, t_long, S_long, '(b) STFT, 1000-sample (1 s) window'),
                                        (f_short, t_short, S_short, '(c) STFT, 16-sample (16 ms) window')]):
    keep = (f >= 2) & (f <= 250); step = max(1, len(t)//1000)
    ax.pcolormesh(t[::step], f[keep], 10*np.log10(S[keep][:, ::step]/S.max() + 1e-12), vmin=-40, vmax=0,
                  cmap='viridis', shading='auto', rasterized=True)
    ax.set_yscale('log'); ax.set_ylim(2, 250); ax.set_xlim(0, 4)
    ax.set_ylabel('frequency (Hz)'); ax.set_title(lab, fontsize=12); ax.grid(False)
axs[2].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

<b>Figure 13.1.</b> One window does not fit all. (a) A test signal: a 4 Hz plus 8 Hz tone pair centered at 1 s and two clicks 30 ms apart at 3 s. (b) Spectrogram with a 1 s window: the two tones are resolved, but the clicks merge into one broadband smear (2 tone peaks, 0 separate clicks). (c) Spectrogram with a 16 ms window: the clicks are separate, but the tones merge into one blob (0 separate tone peaks, 2 clicks). Both use a logarithmic frequency axis and a 40 dB color range. <i>(Python-generated.)</i>

<a name="13.2"></a>
## 13.2 Tiling the Time–Frequency Plane

Every analysis window has a duration $\sigma_t$ and a bandwidth $\sigma_f$, measured as the
root-mean-square widths of $|x(t)|^2$ and $|X(f)|^2$. They cannot both be small:

$$\sigma_t\,\sigma_f \;\geq\; \frac{1}{4\pi} \tag{3}$$

with equality only for a Gaussian. A window can therefore be drawn as a tile of area at least $1/(4\pi)$ in the time–frequency plane, and a
time–frequency transform is a way of covering the plane with tiles. The STFT uses the same tile everywhere. A wavelet transform makes the tiles
short and wide at high frequency and long and narrow at low frequency, keeping their area fixed: the relative bandwidth $\sigma_f/f$ is
constant, so every tile covers the same fraction of an octave.

In [ ]:
C = DSP_COLORS
from matplotlib.patches import Rectangle
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.6))
# (a) STFT: every tile has the same width and height
for i in range(8):
    for j in range(8):
        axa.add_patch(Rectangle((i/8, j/8), 1/8, 1/8, fc=C['passband'], ec=C['window'], lw=1.2))
# (b) dyadic wavelet tiling: the band [2^-(j+1), 2^-j] uses tiles of width 1/(32 * 2^-j), so every tile area is 1/64
tiles_b = []
for j in range(4):
    flo, fhi = 2.0**-(j + 1), 2.0**-j
    dt_tile = (1/64)/(fhi - flo)
    for k in range(int(round(1/dt_tile))):
        tiles_b.append((k*dt_tile, flo, dt_tile, fhi - flo))
tiles_b += [(k/4, 0, 1/4, 1/16) for k in range(4)]   # the lowest band [0, 1/16]: the scaling (approximation) tiles
for (x0, y0, w, h) in tiles_b:
    axb.add_patch(Rectangle((x0, y0), w, h, fc=C['passband'], ec=C['freq_domain'], lw=1.2))
for ax, ttl in [(axa, '(a) STFT: one tile shape at every frequency'), (axb, '(b) Wavelets: narrow in time at high frequency')]:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel('time (fraction of record)'); ax.set_ylabel('frequency (fraction of Nyquist)')
    ax.set_title(ttl, fontsize=12); ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 13.2.</b> Two ways to tile the time–frequency plane with tiles of equal area. (a) The STFT: every tile has the same duration and bandwidth (Module 12). (b) A dyadic wavelet tiling: each octave band uses tiles half as long and twice as tall as the band below it; the bottom strip holds the coarse approximation. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; time–frequency uncertainty.</b> Equation (3) is the scaling theorem of Module 5 (Section 5.6) in another form: squeeze a
signal in time by a factor of two and its spectrum spreads by two, so $\sigma_t\sigma_f$ cannot shrink. Module 12 met it as a choice of window
length (Figure 12.8: 400 vs 200 samples). The wavelet transform does not beat the bound; it changes how the tiles are shared out across frequency.
For the Gaussian, the numerical product is 0.07958, equal to $1/(4\pi)$.
</div>

<a name="13.3"></a>
## 13.3 Wavelets: Little Waves

A <b>wavelet</b> $\psi(t)$ is a short oscillation with zero mean and finite energy,

$$\int_{-\infty}^{\infty} \psi(t)\,\mathrm{d}t = 0 \tag{4}$$

(the admissibility condition, stated here in its practical form). From one mother wavelet we make a whole family by stretching it by a scale
$a > 0$ and shifting it to a time $b$:

$$\psi_{a,b}(t) = \frac{1}{a}\,\psi\!\left(\frac{t - b}{a}\right) \tag{5}$$

Large $a$ gives a long, low-frequency wavelet; small $a$ a short, high-frequency one. The factor $1/a$ (an L1 normalization) is chosen so that a
pure tone produces the same peak value at every scale; Section 13.4.2 explains why this matters.

Two wavelets are used throughout this module. The <b>Ricker</b> (or Mexican-hat) wavelet is the negative second derivative of a Gaussian:

$$\psi(t) = \left(1 - \frac{t^2}{a^2}\right) e^{-t^2/(2a^2)} \quad\Longleftrightarrow\quad \psi(t) = \left(1 - 2\pi^2 f_p^2 t^2\right) e^{-\pi^2 f_p^2 t^2} \tag{6}$$

where the second form is the one used for seismic source wavelets, with peak frequency $f_p$. Its amplitude spectrum is

$$|\Psi(\omega)| \propto (a\omega)^2 e^{-(a\omega)^2/2}, \qquad f_p = \frac{\sqrt{2}}{2\pi a} \tag{7}$$

The <b>Morlet</b> wavelet is a complex exponential under a Gaussian envelope, the Gaussian window of Module 12 applied to $e^{i\omega_0 t}$:

$$\psi(t) = e^{i\omega_0 t}\, e^{-t^2/2}, \qquad \omega_0 = 6, \qquad f_c = \frac{\omega_0}{2\pi a} \tag{8}$$

In [ ]:
C = DSP_COLORS
dtw = 0.0005
tw = np.arange(-0.15, 0.15 + dtw/2, dtw)
fw = np.linspace(0, 100, 1001)
cols = [C['time_domain'], C['freq_domain'], C['impulse_resp']]
fig, axs = plt.subplots(2, 2, figsize=(11, 6.4))
for fp, col in zip([10, 20, 40], cols):
    a = np.sqrt(2)/(2*np.pi*fp)                                   # Ricker scale for peak frequency fp (equation 7)
    axs[0, 0].plot(tw, (1 - tw**2/a**2)*np.exp(-tw**2/(2*a**2)), color=col, label=f'$f_p$ = {fp} Hz')
    aw = a*2*np.pi*fw
    axs[0, 1].plot(fw, aw**2*np.exp(-aw**2/2)/(2*np.exp(-1)), color=col)          # equation (7), peak normalized to 1
    axs[0, 1].axvline(fp, color=col, ls=':', lw=1.2)
    am = 6/(2*np.pi*fp)                                           # Morlet scale for center frequency fp (equation 8)
    env = np.exp(-tw**2/(2*am**2))
    off = {10: 2.4, 20: 0.0, 40: -2.4}[fp]
    axs[1, 0].plot(tw, off + np.cos(6*tw/am)*env, color=col, label=f'$f_c$ = {fp} Hz')
    axs[1, 0].plot(tw, off + env, '--', color=col, lw=1); axs[1, 0].plot(tw, off - env, '--', color=col, lw=1)
    axs[1, 1].plot(fw, np.exp(-(am*2*np.pi*fw - 6)**2/2), color=col)                 # Gaussian centered on f_c
    axs[1, 1].axvline(fp, color=col, ls=':', lw=1.2)
axs[0, 0].set_title('(a) Ricker (Mexican-hat) wavelets', fontsize=12); axs[0, 0].set_xlim(-0.15, 0.15)
axs[0, 0].set_xlabel('time (s)'); axs[0, 0].set_ylabel(r'$\psi(t)$'); axs[0, 0].legend(fontsize=10, loc='upper right')
axs[0, 1].set_title('(b) Their amplitude spectra (peak at $f_p$)', fontsize=12)
axs[1, 0].set_title('(c) Morlet wavelets: real part and envelope', fontsize=12); axs[1, 0].set_xlim(-0.15, 0.15)
axs[1, 0].set_xlabel('time (s)'); axs[1, 0].set_ylabel(r'Re $\psi(t)$ (offset)'); axs[1, 0].set_yticks([])
axs[1, 0].legend(fontsize=10, loc='upper right', ncol=3)
axs[1, 0].set_ylim(-3.9, 4.6)
axs[1, 1].set_title('(d) Their amplitude spectra (peak at $f_c$)', fontsize=12)
for ax in axs[:, 1]:
    ax.set_xlim(0, 100); ax.set_ylim(0, 1.08); ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('normalized amplitude')
plt.tight_layout(); plt.show()

<b>Figure 13.3.</b> The two analyzing wavelets used in this module, each at 10, 20 and 40 Hz. (a) Ricker wavelets, which cross zero at $t = \pm a$ and have side lobes 0.445 times the central peak; (b) their amplitude spectra, which peak at $f_p = \sqrt{2}/(2\pi a)$ (dotted). (c) The real parts of Morlet wavelets (offset vertically) with their Gaussian envelopes (dashed); (d) their spectra, Gaussians centered on $f_c = 6/(2\pi a)$. The Morlet spectra keep the same width relative to their center frequency. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; the seismic wavelet is a wavelet.</b> The Ricker wavelet that built the synthetic seismogram of the convolutional model in Module 7
(Figure 7.1) is the negative second derivative of a Gaussian (MATH 213: differentiate $e^{-t^2/2}$ twice), so it has zero mean and is a valid analyzing
wavelet. A CWT with Ricker wavelets therefore asks, at every time, "how much does the trace look like a seismic pulse of this peak frequency?"
</div>

<div class="alert alert-success">
<h3><center>Distraction A: Why ω₀ = 6?</center></h3>

A Morlet wavelet with a small $\omega_0$ does not have zero mean: the Gaussian envelope leaks energy to zero frequency.
The mean of $e^{i\omega_0 t}e^{-t^2/2}$ is $\sqrt{2\pi}\,e^{-\omega_0^2/2}$, so relative to its peak spectral value it is $e^{-\omega_0^2/2}$, which is 1.5e-08 for $\omega_0 = 6$: small enough to ignore. The FFT version used here goes one step further and sets
the negative frequencies to zero, which makes it exactly admissible. Larger $\omega_0$ gives more oscillations under the envelope, better frequency
resolution and worse time resolution: $\omega_0$ is a knob on the tile shape of Figure 13.2.
</div>

<a name="13.4"></a>
## 13.4 The Continuous Wavelet Transform

The <b>continuous wavelet transform</b> of a signal $x(t)$ compares it with every member of
the wavelet family:

$$W(a,b) = \int_{-\infty}^{\infty} x(t)\,\psi_{a,b}^{*}(t)\,\mathrm{d}t \tag{9}$$

For each scale $a$, $W(a,\cdot)$ is the crosscorrelation of the signal with the scaled wavelet (Module 7, Section 7.8), so it is large wherever the
signal locally resembles that wavelet. The magnitude $|W(a,b)|$, plotted against shift $b$ (time) and frequency $f = f_c/a$, is the
<b>scalogram</b>, the wavelet counterpart of the spectrogram.

<a name="13.4.1"></a>
### 13.4.1 Computing the CWT with the FFT

By the correlation theorem, each row of the CWT is one inverse FFT:

$$W(a,\cdot) = \mathcal{F}^{-1}\left[\,X(\omega)\,\Psi^{*}(a\omega)\,\right] \tag{10}$$

The cell below is the whole CWT: it pads the record with zeros, takes one FFT of the data, and for each scale multiplies by the conjugate wavelet
spectrum and inverts. <i>Toggle the code on (button at the top of the notebook) to read this cell; it is part of the content.</i>

In [ ]:
# The continuous wavelet transform in 15 lines (Section 13.4.1)
def ricker_hat(w):
    """Fourier transform of the Mexican-hat wavelet (1 - t^2) exp(-t^2/2), up to a constant."""
    return w**2*np.exp(-w**2/2)

def morlet_hat(w, w0=6.0):
    """Fourier transform of the analytic Morlet wavelet exp(i w0 t) exp(-t^2/2); negative frequencies set to zero."""
    return np.exp(-(w - w0)**2/2)*(w > 0)

def cwt(x, dt, scales, wavelet='morlet', w0=6.0):
    """W(a,b) = (1/a) * integral x(t) psi*((t-b)/a) dt for each scale a, computed as the FFT product of equation (10)."""
    n = len(x)
    npad = int(2**np.ceil(np.log2(2*n)))            # zero-pad so the circular correlation equals the linear one
    X = np.fft.fft(x, npad)
    w = 2*np.pi*np.fft.fftfreq(npad, d=dt)          # angular frequency of each FFT bin
    W = np.empty((len(scales), n), dtype=np.complex64)
    for i, a in enumerate(scales):
        P = ricker_hat(a*w)/ricker_hat(np.sqrt(2)) if wavelet == 'ricker' else morlet_hat(a*w, w0)
        W[i] = np.fft.ifft(X*np.conj(P))[:n]
    return W

def scale_to_freq(a, wavelet='morlet', w0=6.0):
    """Frequency (Hz) at which a wavelet of scale a (s) peaks: sqrt(2)/(2 pi a) for Ricker, w0/(2 pi a) for Morlet."""
    return (np.sqrt(2) if wavelet == 'ricker' else w0)/(2*np.pi*np.asarray(a, float))

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> Each row of the scalogram is the crosscorrelation of the data with one scaled wavelet, the
operation of Module 7. The correlation theorem turns it into a product of spectra, $X(\omega)\Psi^{*}(a\omega)$ in equation (10), which is why
fifteen lines of FFT code compute the whole CWT. Zero-padding to at least twice the record length makes the circular correlation of the DFT equal
the linear one (Module 11, Section 11.10). Checked against a direct sum, the FFT version agrees to 3e-08 (the rows are stored in single precision).
</div>

Figure 13.4 builds three rows of the scalogram of the test signal one shift at a time. Watch the red wavelet in panel (a): at 4 Hz
and 8 Hz it is long, spans several oscillations of the tones, and responds strongly only near $t = 1$ s, where its row (blue or red in panel b)
rises to a single broad hump. At 120 Hz the wavelet is a few milliseconds long, ignores the slow tones entirely, and its row (green) stays near
zero until it reaches the clicks at 3 s, where it produces two sharp spikes.

In [ ]:
C = DSP_COLORS
# . . Build three rows of the scalogram of the test signal one shift at a time
f_rows = np.array([4.0, 8.0, 120.0])                      # three analysis frequencies (Hz)
a_rows = 6/(2*np.pi*f_rows)                               # their Morlet scales (s)
W_rows = np.abs(cwt(x_test, 1/fs_test, a_rows))           # the full rows, computed once
b_steps = np.linspace(0.4, 4.0, 10)                       # reveal each row up to these shifts (s)
frames = [(i, b) for i in range(3) for b in b_steps]      # 3 scales x 10 shifts = 30 frames
n_frames = len(frames)
row_cols = [C['time_domain'], C['freq_domain'], C['impulse_resp']]

fig, (axa, axb) = plt.subplots(2, 1, figsize=(9.0, 5.0), sharex=True)
axa.plot(t_test, x_test, color=C['continuous'], lw=0.7)
wv, = axa.plot([], [], color=C['output'], lw=1.6)
axa.set_ylim(-4.5, 6.5); axa.set_ylabel('amplitude'); axa.set_title('(a) Test signal (gray) and the scaled, shifted Morlet wavelet (red, real part)', fontsize=11)
lines = [axb.plot([], [], color=c, lw=1.6, label=f'{f:.0f} Hz row')[0] for c, f in zip(row_cols, f_rows)]
dot, = axb.plot([], [], 'o', color='k', ms=6)
axb.set_xlim(0, 4); axb.set_ylim(0, 1.15*W_rows.max()); axb.set_xlabel('shift $b$ (s)')
axb.set_ylabel(r'$|W(a,b)|$'); axb.set_title('(b) Rows of the scalogram, revealed up to the current shift', fontsize=11)
axb.legend(fontsize=9, loc='upper left')
title = fig.suptitle('', fontsize=12)

def update(k):
    i, b = frames[k]
    a = a_rows[i]
    bc = min(b, 3.9)                                          # keep the drawn wavelet inside the window
    tt = t_test[np.abs(t_test - bc) < 4*a]
    wv.set_data(tt, 3*np.cos(6*(tt - bc)/a)*np.exp(-(tt - bc)**2/(2*a**2)))
    for j, ln in enumerate(lines):
        upto = 4.0 if j < i else (b if j == i else -1.0)
        m = t_test <= upto
        ln.set_data(t_test[m], W_rows[j][m])
    idx = min(int(round(b*fs_test)), len(t_test) - 1)
    dot.set_data([t_test[idx]], [W_rows[i][idx]])
    title.set_text(f'scale a = {a:.4f} s (f = {f_rows[i]:.0f} Hz),  shift b = {b:.2f} s')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=500, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 13.4.</b> Building the CWT row by row (30 frames: ten shifts at each of three scales). (a) The test signal of Figure 13.1 (gray) and the real part of the Morlet wavelet at the current scale and shift (red). (b) The magnitude $|W(a,b)|$ for $f$ = 4, 8 and 120 Hz, revealed up to the current shift $b$; the last frame shows the three complete rows. <i>(Python-generated.)</i>

Computing all 120 rows between 2 and 250 Hz gives the full scalogram in Figure 13.5. It passes both tests that each spectrogram in
Figure 13.1 failed.

In [ ]:
C = DSP_COLORS
f_scal = np.geomspace(2, 250, 120)
W_test = np.abs(cwt(x_test, 1/fs_test, 6/(2*np.pi*f_scal)))
fig = plt.figure(figsize=(12, 6.2))
gsp = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])
axa = fig.add_subplot(gsp[0, :]); axb = fig.add_subplot(gsp[1, 0]); axc = fig.add_subplot(gsp[1, 1])
step = 4
axa.pcolormesh(t_test[::step], f_scal, W_test[:, ::step]/W_test.max(), cmap='viridis', shading='auto', rasterized=True)
axa.set_yscale('log'); axa.set_ylim(2, 250); axa.set_xlabel('time (s)'); axa.set_ylabel('frequency (Hz)')
axa.set_title('(a) Morlet scalogram $|W|$ of the test signal', fontsize=12); axa.grid(False)
i1 = int(1.0*fs_test)
axb.semilogx(f_scal, W_test[:, i1], color=C['freq_domain'])
axb.set_xlim(2, 30); axb.set_xlabel('frequency (Hz)'); axb.set_ylabel(r'$|W|$')
from matplotlib.ticker import NullFormatter
axb.set_xticks([2, 4, 8, 16, 30]); axb.set_xticklabels(['2', '4', '8', '16', '30']); axb.xaxis.set_minor_formatter(NullFormatter())
axb.set_title('(b) Slice at t = 1 s: two tone peaks', fontsize=12)
hb = f_scal > 100; seg = slice(int(2.95*fs_test), int(3.08*fs_test))
axc.plot(1000*(t_test[seg] - 3.0), W_test[hb][:, seg].sum(axis=0), color=C['time_domain'])
axc.set_xlabel('time after 3 s (ms)'); axc.set_ylabel(r'$\sum_{f>100\,\mathrm{Hz}} |W|$')
axc.set_title('(c) Slice above 100 Hz: two click peaks', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 13.5.</b> (a) Morlet scalogram of the test signal of Figure 13.1, 120 logarithmically spaced frequencies from 2 to 250 Hz. (b) The slice at $t$ = 1 s has two peaks, at 4.0 and 7.9 Hz: the tones are resolved. (c) Summed over frequencies above 100 Hz, the scalogram has two peaks 30 ms apart: the clicks are resolved too. <i>(Python-generated.)</i>

<a name="13.4.2"></a>
### 13.4.2 From scale to frequency

For the Morlet wavelet at scale $a$ the time and frequency widths are

$$\sigma_t = \frac{a}{\sqrt{2}}, \qquad \sigma_f = \frac{1}{2\sqrt{2}\,\pi a} \tag{11}$$

so their product is the Gaussian minimum $1/(4\pi)$ at every scale, and the relative bandwidth $\sigma_f/f_c = 1/(\sqrt{2}\,\omega_0)$ is the same
for every scale. This is the constant-Q property drawn in Figure 13.2(b).

The normalization $1/a$ in equation (5) puts the ridge of a pure tone exactly at $f = f_c/a$: a 25 Hz cosine gives a scalogram maximum at 25.03 Hz with the Morlet wavelet and 25.03 Hz with the Ricker wavelet. With the other common
normalization, $1/\sqrt{a}$, the ridge of a tone is biased toward low frequencies, because longer wavelets collect more energy.

<div class="alert alert-info">
<b>Note:</b> A Morlet wavelet at every scale has the same quality factor $Q = f_c/\sigma_f$, like a bank of RLC band-pass circuits (PHGN 200) tuned
an octave apart, each with the same $Q$: the higher the center frequency, the wider the passband and the faster the circuit rings down.
</div>

<div class="alert alert-warning">
<b>Common misconception:</b> "A larger scale means a higher frequency." It is the other way round: frequency is inversely proportional to scale,
$f = f_c/a$. A large scale is a long, stretched wavelet and responds to low frequencies; this is why scalograms are usually plotted against
frequency rather than scale.
</div>

<a name="13.5"></a>
## 13.5 CWT Examples

Figure 13.6 returns to the chirps of Module 12 (Figure 12.1; same parameters: 10 to 200 Hz over 16.384 s,
2000 samples/s). The spectrogram uses Module 12's 1024-sample Hann window. The scalogram follows the instantaneous frequency equally well, but its
time resolution improves as the frequency rises.

In [ ]:
C = DSP_COLORS
# . . Module 12's chirps (same parameters)
nt, dt = 2048*16, 0.0005
f0, f1 = 10, 200
t1 = nt*dt
t_chirp = np.arange(nt)*dt
CC_LI = signal.chirp(t_chirp, f0=f0, t1=t1, f1=f1, method='linear')
CC_HY = signal.chirp(t_chirp, f0=f0, t1=t1, f1=f1, method='hyperbolic')
finst = {'linear': f0 + (f1 - f0)*t_chirp/t1,
         'hyperbolic': f0*f1*t1/((f0 - f1)*t_chirp + f1*t1)}
f_ch = np.geomspace(5, 300, 80)
fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
W_ch = {}
for row, (name, xc) in enumerate([('linear', CC_LI), ('hyperbolic', CC_HY)]):
    fS, tS, S = signal.spectrogram(xc, 1/dt, window='hann', nperseg=1024, noverlap=896)
    keep = (fS >= 5) & (fS <= 300)
    A = np.sqrt(S[keep]); axs[row, 0].pcolormesh(tS, fS[keep], A/A.max(), cmap='viridis', shading='auto', rasterized=True)
    W = np.abs(cwt(xc, dt, 6/(2*np.pi*f_ch))); W_ch[name] = W
    step = 32
    axs[row, 1].pcolormesh(t_chirp[::step], f_ch, W[:, ::step]/W.max(), cmap='viridis', shading='auto', rasterized=True)
    for ax in axs[row]:
        ax.plot(t_chirp[::256], finst[name][::256], '--', color='white', lw=1.0)
    axs[row, 0].set_title(f'({"ac"[row]}) STFT, {name} chirp', fontsize=12)
    axs[row, 1].set_title(f'({"bd"[row]}) Morlet scalogram, {name} chirp', fontsize=12)
for ax in axs.flat:
    ax.set_yscale('log'); ax.set_ylim(5, 300); ax.set_xlim(0, t1); ax.grid(False)
for ax in axs[:, 0]:
    ax.set_ylabel('frequency (Hz)')
for ax in axs[1]:
    ax.set_xlabel('time (s)')
plt.tight_layout(); plt.show()

<b>Figure 13.6.</b> (a) Spectrogram and (b) Morlet scalogram of the linear chirp of Module 12; (c) spectrogram and (d) scalogram of the hyperbolic chirp. Dashed white: the instantaneous frequency. Both transforms are normalized to their own maximum and plotted on a logarithmic frequency axis. The scalogram ridge stays within 2.5% (linear) and 2.1% (hyperbolic) of the instantaneous frequency at 2, 6, 10 and 14 s. The spectrogram's frequency resolution is the same at every frequency, so at low frequency, on the log axis, its bins look coarse. <i>(Python-generated.)</i>

The Tohoku record of Module 12 is a better test, because it contains a sharp body-wave arrival and long-period surface waves spread
over hours. Figure 13.7 uses the same data (vertical ground velocity at station GR.WET in Germany, in mm/s), decimated from 20 to 4 samples/s
with a zero-phase anti-alias filter (Module 10) so the CWT stays small.

In [ ]:
C = DSP_COLORS
# . . Module 12's Tohoku record (vertical velocity, mm/s, 20 samples/s), decimated to 4 samples/s for the CWT
tohoku = np.load('data/Tohoku.npz')['tohoku'].astype(float)*1000
x4 = signal.decimate(tohoku, 5, ftype='fir', zero_phase=True)     # zero-phase anti-alias filter, then every 5th sample
dt4 = 0.25
i0, i1 = int(2500/dt4), int(20000/dt4)
seg_t = x4[i0:i1]; t_seg = np.arange(i0, i1)*dt4                   # seconds after 05:00:00 UTC
env_t = np.abs(signal.hilbert(seg_t))
t_first = t_seg[np.argmax(env_t > 10*np.median(env_t[:int(600/dt4)]))]   # first sample 10x above the pre-event level
f_toh = np.geomspace(0.005, 1.0, 60)
W_toh = np.abs(cwt(seg_t, dt4, 6/(2*np.pi*f_toh)))
fig, (axa, axb) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.6]))
axa.plot(t_seg, seg_t, color=C['time_domain'], lw=0.4)
axa.axvline(t_first, color=C['output'], ls='--', lw=1.2)
axa.text(t_first + 150, 0.8*np.abs(seg_t).max(), 'P', color=C['output'], fontsize=12)
axa.set_xlim(2500, 20000); axa.set_ylabel('velocity (mm/s)')
axa.set_title('(a) Tohoku earthquake, vertical velocity at GR.WET (decimated to 4 samples/s)', fontsize=12)
step = 50
axb.pcolormesh(t_seg[::step], f_toh, np.log10(W_toh[:, ::step] + 1e-12), cmap='viridis', shading='auto', rasterized=True,
               vmin=np.log10(W_toh.max()) - 3, vmax=np.log10(W_toh.max()))
axb.axvline(t_first, color='white', ls='--', lw=1.0)
axb.annotate('dispersed surface waves', xy=(5600, 0.03), xytext=(9000, 0.25), color='white', fontsize=11,
             arrowprops=dict(arrowstyle='->', color='white'))
axb.set_yscale('log'); axb.set_ylim(0.005, 1.0); axb.set_xlabel('time after 05:00:00 UTC (s)'); axb.set_ylabel('frequency (Hz)')
axb.set_title(r'(b) Morlet scalogram, $\log_{10}|W|$ (3 decades shown)', fontsize=12); axb.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 13.7.</b> (a) The 2011 Tohoku earthquake recorded at GR.WET, Germany, 2500–20,000 s after 05:00:00 UTC; the dashed line marks the first arrival, where the envelope first exceeds ten times its pre-event level (3566 s). (b) Its Morlet scalogram, $\log_{10}|W|$, from 0.005 to 1 Hz. The long-period surface waves arrive first and the shorter periods later: the surface waves are dispersed. <i>(Data: GR.WET BHZ, BGR; course file data/Tohoku.npz.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; dispersion in a scalogram.</b> Compare with the Tohoku spectrogram in Module 12 (Figure 12.6): the scalogram resolves the long-period
surface waves at the bottom of the plot and still marks the P arrival sharply at the top. The surface waves are dispersed because their speed depends
on period: longer periods sense deeper, faster parts of the mantle (POTE1, GPGN 328), so they arrive first. Microquestion 13.1 turns the scalogram
into group velocities.
</div>

<a name="13.6"></a>
## 13.6 The Discrete Wavelet Transform: the Haar Filter Bank

The CWT is highly redundant: a record of $N$ samples analyzed at
120 scales becomes $120N$ numbers. The <b>discrete wavelet transform</b> keeps exactly $N$ numbers and can be inverted exactly. It is built from a
pair of filters, a low-pass $h$ and a high-pass $g$, each followed by keeping every second sample (Figure 13.8). The low-pass branch gives a coarse
<b>approximation</b> $a[n]$, the high-pass branch the <b>details</b> $d[n]$, and the whole operation is repeated on the approximation.

In [ ]:
C = DSP_COLORS
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(11, 3.6))
def box(x, y, txt, fc):
    ax.add_patch(FancyBboxPatch((x - 0.42, y - 0.3), 0.84, 0.6, boxstyle='round,pad=0.05', fc=fc, ec='k', lw=1.2))
    ax.text(x, y, txt, ha='center', va='center', fontsize=13)
def arrow(x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', lw=1.4, color='k'))
yh, yg = 1.0, -1.0
ax.text(0.0, 0.0, '$x[n]$', ha='center', va='center', fontsize=14)
for y, hf, sf, coef in [(yh, '$h$', r'$\tilde h$', '$a[n]$'), (yg, '$g$', r'$\tilde g$', '$d[n]$')]:
    arrow(0.35, 0.0, 1.0, y); box(1.5, y, hf, C['passband'])
    arrow(1.95, y, 2.6, y); box(3.1, y, r'$\downarrow 2$', 'white')
    arrow(3.55, y, 4.45, y); ax.text(5.0, y + 0.42, coef, ha='center', fontsize=13)
    arrow(4.55, y, 5.45, y)
    box(5.95, y, r'$\uparrow 2$', 'white'); arrow(6.4, y, 7.05, y); box(7.55, y, sf, C['passband'])
    arrow(8.0, y, 8.75, 0.0)
ax.add_patch(plt.Circle((9.0, 0.0), 0.25, fc='white', ec='k', lw=1.2)); ax.text(9.0, 0.0, r'$+$', ha='center', va='center', fontsize=14)
arrow(9.25, 0.0, 9.9, 0.0); ax.text(10.3, 0.0, r'$\hat x[n]$', ha='center', va='center', fontsize=14)
ax.text(2.3, 1.75, 'analysis', ha='center', fontsize=12, color=C['neutral'])
ax.text(6.75, 1.75, 'synthesis', ha='center', fontsize=12, color=C['neutral'])
ax.axvline(4.95, ymin=0.05, ymax=0.95, color=C['neutral'], ls=':', lw=1)
ax.set_xlim(-0.4, 10.75); ax.set_ylim(-1.55, 2.05); ax.axis('off')
plt.tight_layout(); plt.show()

<b>Figure 13.8.</b> A two-channel filter bank. Analysis (left): the signal is filtered by a low-pass $h$ and a high-pass $g$ and each output is downsampled by two, giving the approximation $a[n]$ and the details $d[n]$. Synthesis (right): each channel is upsampled by two (zeros inserted), filtered and added. For the filters of this section, $\hat x[n] = x[n]$ exactly. <i>(Python-generated.)</i>

<a name="13.6.1"></a>
### 13.6.1 Averages and differences

The simplest pair is the Haar filter bank: scaled averages and differences of neighboring samples,

$$a[n] = \frac{x[2n] + x[2n+1]}{\sqrt{2}}, \qquad d[n] = \frac{x[2n] - x[2n+1]}{\sqrt{2}} \tag{12}$$

Adding and subtracting recovers every sample:

$$x[2n] = \frac{a[n] + d[n]}{\sqrt{2}}, \qquad x[2n+1] = \frac{a[n] - d[n]}{\sqrt{2}} \tag{13}$$

In general the analysis is a pair of filters followed by downsampling, with the high-pass filter built from the low-pass one:

$$a[n] = \sum_k h[k]\,x[2n+k], \qquad d[n] = \sum_k g[k]\,x[2n+k], \qquad g[k] = (-1)^k\,h[L-1-k] \tag{14}$$

where $L$ is the filter length and the signal is extended periodically at its ends. The cell below implements equations (14) and its inverse
for any such pair. <i>Toggle the code on to read it.</i>

In [ ]:
# The discrete wavelet transform in 20 lines (Section 13.6)
s3 = np.sqrt(3)
FILT = {'haar': np.array([1, 1])/np.sqrt(2),                                    # equation (12)
        'db4':  np.array([1 + s3, 3 + s3, 3 - s3, 1 - s3])/(4*np.sqrt(2))}      # equation (15)

def qmf(h):
    """High-pass partner g[k] = (-1)^k h[L-1-k] of the low-pass filter h (equation 14)."""
    return h[::-1]*(-1)**np.arange(len(h))

def dwt_step(x, h):
    """One level: filter with h and g and keep every second output (periodic extension at the ends)."""
    n = len(x); idx = (2*np.arange(n//2)[:, None] + np.arange(len(h))[None, :]) % n
    return x[idx] @ h, x[idx] @ qmf(h)

def idwt_step(a, d, h):
    """Undo one level: the transpose of the orthonormal analysis step."""
    n = 2*len(a); x = np.zeros(n)
    idx = (2*np.arange(n//2)[:, None] + np.arange(len(h))[None, :]) % n
    np.add.at(x, idx, a[:, None]*h[None, :] + d[:, None]*qmf(h)[None, :])
    return x

def wavedec(x, h, levels):
    """Multilevel DWT: the coarsest approximation and the details [d_J, ..., d_1]."""
    ds, a = [], np.asarray(x, float)
    for _ in range(levels):
        a, d = dwt_step(a, h); ds.append(d)
    return a, ds[::-1]

def waverec(a, ds, h):
    """Inverse multilevel DWT."""
    for d in ds:
        a = idwt_step(a, d, h)
    return a

<div class="alert alert-info">
<b>Proof:</b> <i>Why the Haar step loses nothing.</i> Equation (12) maps each pair $(x[2n], x[2n+1])$ to $(a[n], d[n])$ by the matrix
$\frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$, whose columns are orthonormal: it is a rotation (and reflection) by $45^\circ$.
A rotation preserves length, so $a[n]^2 + d[n]^2 = x[2n]^2 + x[2n+1]^2$, and its inverse is its transpose, which is equation (13).
Repeating the step on the approximations keeps the whole transform orthonormal (Module 8: $\mathbf{W}^T\mathbf{W} = \mathbf{I}$).
</div>

Figure 13.9 applies the Haar step six times to a 256-sample trace made of two blocks and a short chirp burst. At each level the
approximation (blue, drawn as block averages in panel a) halves in length and gets coarser, and one more row of details appears in panel (b).
Watch where the energy goes: the blocks are captured by a few large details at their edges and by the coarse approximation, while the chirp burst
lives in the fine details $d_1$–$d_3$.

In [ ]:
C = DSP_COLORS
# . . A 256-sample trace: blocks plus a short chirp burst, decomposed one Haar level per frame
N_h = 256; n_h = np.arange(N_h)
x_h = np.where((n_h >= 40) & (n_h < 100), 1.5, 0.0) + np.where((n_h >= 100) & (n_h < 130), -0.8, 0.0)
burst = (n_h >= 170) & (n_h < 230)
x_h = x_h + burst*np.sin(2*np.pi*(0.02*(n_h - 170) + 0.004*(n_h - 170)**2))
levels_h = 6
n_frames = levels_h + 1                                  # frame 0 = original, frame j = after j levels
approx, details = [x_h.copy()], []
a = x_h.copy()
for j in range(levels_h):
    a, d = dwt_step(a, FILT['haar']); approx.append(a); details.append(d)
E_x = np.sum(x_h**2)

fig, (axa, axb) = plt.subplots(2, 1, figsize=(8.6, 5.0), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.4]))
axa.plot(n_h, x_h, color=C['continuous'], lw=1.0, label='original $x[n]$')
ap, = axa.step([], [], where='post', color=C['time_domain'], lw=1.8, label='approximation')
axa.set_ylim(-2.0, 2.4); axa.set_ylabel('amplitude'); axa.legend(loc='upper right', fontsize=9)
axa.set_title('(a) Approximation at the current level, drawn as block averages', fontsize=11)
dlines = [axb.step([], [], where='post', color=C['freq_domain'], lw=1.2)[0] for _ in range(levels_h)]
dlabels = [axb.text(258, -1.6*(j + 1), '', fontsize=9, va='center') for j in range(levels_h)]
axb.set_xlim(0, 280); axb.set_ylim(-1.6*levels_h - 1.0, 0.8); axb.set_yticks([])
axb.set_xlabel('sample index $n$ (detail coefficients drawn at their sample position)')
axb.set_title('(b) Detail coefficients $d_1, d_2, \\ldots$ (each row offset)', fontsize=11)
title = fig.suptitle('', fontsize=12)

def update(k):
    a = approx[k]; m = 2**k
    ap.set_data(np.arange(0, N_h + 1, m), np.append(a/np.sqrt(m), a[-1]/np.sqrt(m)))
    for j in range(levels_h):
        if j < k:
            d = details[j]; mj = 2**(j + 1)
            dlines[j].set_data(np.arange(0, N_h + 1, mj), np.append(d, d[-1])/np.sqrt(mj) - 1.6*(j + 1))
            dlabels[j].set_text(f'$d_{j + 1}$ ({len(d)})')
        else:
            dlines[j].set_data([], []); dlabels[j].set_text('')
    Ed = sum(np.sum(d**2) for d in details[:k])
    title.set_text(f'level {k}: {N_h//m} approximation samples;  energy in details = {100*Ed/E_x:.1f}%')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=900, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 13.9.</b> A six-level Haar decomposition, one level per frame (7 frames). (a) The original trace (gray) and the approximation at the current level, drawn as block averages (blue). (b) The detail coefficients $d_1, d_2, \ldots$ (offset rows, with their lengths). The title tracks the share of the energy in the details. Reconstructing from the last frame recovers the trace to 2e-15. <i>(Python-generated.)</i>

<a name="13.6.2"></a>
### 13.6.2 Downsampling by two without losing information

Keeping every second sample of a signal halves its Nyquist frequency
(Module 10), so each channel on its own is aliased. Together they are not: the two channels hold $N/2 + N/2 = N$ numbers for $N$ samples, and
because the transform is orthonormal it preserves energy,

$$\sum_n x[n]^2 = \sum_n a_J[n]^2 + \sum_{j=1}^{J} \sum_n d_j[n]^2 \tag{15}$$

Parseval's theorem for the DWT (MATH 348; Module 11). For a random 1024-sample signal and six levels the reconstruction error is 2e-15 (Haar) and 4e-15 (Daubechies-4).

<div class="alert alert-info">
<b>Synthesis &mdash; downsampling by two.</b> Keeping every second sample halves the Nyquist frequency (Module 10, Section 10.5), so each half-band channel
aliases on its own. The high-pass channel carries exactly the information the low-pass channel folds over, and the synthesis filters cancel the
aliasing: the reconstruction error is set only by rounding.
</div>

<a name="13.6.3"></a>
### 13.6.3 Daubechies-4: a smoother filter pair

The Haar filters are short but their frequency responses overlap a lot
(Figure 13.10). Ingrid Daubechies found the shortest orthonormal filters with smoother responses; the four-tap Daubechies-4 low-pass filter is

$$h = \frac{1}{4\sqrt{2}}\left[\,1+\sqrt{3},\; 3+\sqrt{3},\; 3-\sqrt{3},\; 1-\sqrt{3}\,\right] \tag{16}$$

Its high-pass partner has two <b>vanishing moments</b> ($\sum_k g[k] = \sum_k k\,g[k] = 0$), so it ignores not only constants but also straight
lines; the Haar high-pass filter has one.

In [ ]:
C = DSP_COLORS
wz = np.linspace(0, np.pi, 513)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (name, lab) in zip(axs, [('haar', '(a) Haar'), ('db4', '(b) Daubechies-4')]):
    h = FILT[name]; gq = qmf(h); k = np.arange(len(h))
    H = np.exp(-1j*np.outer(wz, k)) @ h; G = np.exp(-1j*np.outer(wz, k)) @ gq
    ax.plot(wz/np.pi, np.abs(H)**2, color=C['time_domain'], label=r'$|H|^2$ (low-pass $h$)')
    ax.plot(wz/np.pi, np.abs(G)**2, color=C['freq_domain'], label=r'$|G|^2$ (high-pass $g$)')
    ax.plot(wz/np.pi, np.abs(H)**2 + np.abs(G)**2, '--', color='k', lw=1.2, label=r'$|H|^2 + |G|^2$')
    ax.set_xlim(0, 1); ax.set_ylim(0, 2.3); ax.set_xlabel(r'frequency $\omega/\pi$ (1 = Nyquist)')
    ax.set_title(lab + f' ({len(h)} taps)', fontsize=12); ax.legend(fontsize=9, loc='center right')
axs[0].set_ylabel('power response')
plt.tight_layout(); plt.show()

<b>Figure 13.10.</b> Power responses of the low-pass $h$ (blue) and high-pass $g$ (red) filters for (a) Haar and (b) Daubechies-4. Both pairs are half-band filters that cross at $\omega = \pi/2$, and both are power complementary: $|H|^2 + |G|^2 = 2$ at every frequency (dashed). The Daubechies-4 responses fall off more steeply. <i>(Python-generated.)</i>

<div class="alert alert-success">
<h3><center>Distraction B: Ingrid Daubechies and compact wavelets</center></h3>

The first wavelet basis, the Haar functions, was published by Alfréd Haar in 1910.
The continuous transform was developed in the early 1980s by Jean Morlet, a geophysicist analyzing seismic reflection data at Elf Aquitaine
(Morlet et al., 1982), with the physicist Alex Grossmann (Grossmann and Morlet, 1984). In 1988 Ingrid Daubechies constructed orthonormal
wavelets of finite length, which made fast, exact discrete transforms practical; the JPEG 2000 image standard (ISO/IEC 15444-1) uses a
wavelet transform. Her book <i>Ten Lectures on Wavelets</i> (Daubechies, 1992) is the classic reference.

<b>References.</b> Haar, A. (1910), Zur Theorie der orthogonalen Funktionensysteme, <i>Mathematische Annalen</i> 69(3), 331&ndash;371.
Morlet, J., G. Arens, E. Fourgeau and D. Giard (1982), Wave propagation and sampling theory, Parts I and II, <i>Geophysics</i> 47(2),
203&ndash;221 and 222&ndash;236. Grossmann, A., and J. Morlet (1984), Decomposition of Hardy functions into square integrable wavelets of
constant shape, <i>SIAM Journal on Mathematical Analysis</i> 15(4), 723&ndash;736. Daubechies, I. (1988), Orthonormal bases of compactly
supported wavelets, <i>Communications on Pure and Applied Mathematics</i> 41(7), 909&ndash;996. Daubechies, I. (1992), <i>Ten Lectures on
Wavelets</i>, SIAM.
</div>

<a name="13.7"></a>
## 13.7 Applications: Denoising and Compression

<a name="13.7.1"></a>
### 13.7.1 Denoising by thresholding

Random noise spreads evenly over all wavelet coefficients, while a signal made of smooth
pieces and sharp edges concentrates in a few large ones. Setting the small coefficients to zero therefore removes mostly noise. With $\lambda$ a
threshold, <i>hard</i> thresholding keeps $d$ if $|d| > \lambda$ and sets it to zero otherwise, and <i>soft</i> thresholding also shrinks the kept
coefficients by $\lambda$. A standard choice is the universal threshold of Donoho and Johnstone (1994; <i>Biometrika</i> 81(3), 425&ndash;455),

$$\lambda = \hat\sigma\sqrt{2\ln N}, \qquad \hat\sigma = \frac{\mathrm{median}\,|d_1|}{0.6745} \tag{17}$$

where $\hat\sigma$ estimates the noise level from the finest details (0.6745 makes the median of $|d|$ equal $\sigma$ for Gaussian noise).

In [ ]:
C = DSP_COLORS
rng = np.random.default_rng(404)
N_d = 1024; n_d = np.arange(N_d)
clean = np.zeros(N_d); clean[150:300] = 2; clean[300:420] = -1; clean[600:620] = 3; clean[700:900] = 1
noisy = clean + 0.4*rng.standard_normal(N_d)

def denoise(x, h, levels, mode='hard'):
    a, ds = wavedec(x, h, levels)
    sig = np.median(np.abs(ds[-1]))/0.6745                 # noise level from the finest details (equation 17)
    lam = sig*np.sqrt(2*np.log(len(x)))                    # universal threshold
    if mode == 'hard':
        th = lambda d: d*(np.abs(d) > lam)
    else:
        th = lambda d: np.sign(d)*np.maximum(np.abs(d) - lam, 0)
    return waverec(a, [th(d) for d in ds], h), sig, lam

rms = lambda y: np.sqrt(np.mean((y - clean)**2))
ma = {L: np.convolve(noisy, np.ones(L)/L, mode='same') for L in [5, 9, 15, 25, 41]}
L_best = min(ma, key=lambda L: rms(ma[L]))
x_haar, sig_hat, lam_u = denoise(noisy, FILT['haar'], 6, 'hard')
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(n_d, noisy, color=C['noise'], lw=0.7, label='noisy')
axs[0].plot(n_d, clean, color='k', lw=1.4, label='clean')
axs[0].set_title(f'(a) Blocks plus Gaussian noise ($\\sigma$ = 0.4); rms error {rms(noisy):.3f}', fontsize=12)
axs[1].plot(n_d, clean, color='k', lw=1.0); axs[1].plot(n_d, ma[L_best], color=C['time_domain'], lw=1.2)
axs[1].set_title(f'(b) Best moving average ({L_best} samples); rms error {rms(ma[L_best]):.3f}', fontsize=12)
axs[2].plot(n_d, clean, color='k', lw=1.0); axs[2].plot(n_d, x_haar, color=C['freq_domain'], lw=1.2)
axs[2].set_title(f'(c) Haar, hard threshold $\\lambda$ = {lam_u:.2f}; rms error {rms(x_haar):.3f}', fontsize=12)
axs[0].legend(fontsize=9, loc='upper right', ncol=2)
for ax in axs:
    ax.set_ylim(-2.4, 4.4); ax.set_ylabel('amplitude')
axs[2].set_xlabel('sample index $n$'); axs[2].set_xlim(0, N_d)
plt.tight_layout(); plt.show()

<b>Figure 13.11.</b> Denoising a blocky trace. (a) The clean trace (black) and the trace plus Gaussian noise of standard deviation 0.4 (olive). (b) The best moving average (length 9, chosen from 5, 9, 15, 25 and 41 samples). (c) Six-level Haar transform with hard thresholding at $\lambda$ = 1.40, from the estimated noise level $\hat\sigma$ = 0.376. The wavelet result has the smaller rms error (0.142 vs 0.209) and keeps the edges sharp: the step at sample 150 rises in 2 samples, against 7 for the moving average. <i>(Python-generated.)</i>

For this trace, soft thresholding (0.213) and the smoother Daubechies-4 filters (0.184) both do worse than Haar with hard thresholding,
because the signal really is made of flat blocks, exactly the shape of the Haar basis functions. The best basis depends on the signal.

<a name="13.7.2"></a>
### 13.7.2 Compressing a 2D image

A 2D DWT applies the 1D transform to every row and then to every column, and repeats on the
low-low quarter. The cell below does this for the New York photograph of Module 6 (reduced to 256 × 512 pixels) and keeps only the largest
coefficients.

In [ ]:
# 2D DWT: transform every row, then every column, then repeat on the low-low quarter
def dwt2(A, h, levels):
    Cc = np.array(A, float); r, c = Cc.shape
    for _ in range(levels):
        rows = np.array([np.concatenate(dwt_step(v, h)) for v in Cc[:r, :c]])
        Cc[:r, :c] = np.array([np.concatenate(dwt_step(v, h)) for v in rows.T]).T
        r //= 2; c //= 2
    return Cc

def idwt2(Cc, h, levels):
    A = np.array(Cc, float); r, c = A.shape[0] >> (levels - 1), A.shape[1] >> (levels - 1)
    for _ in range(levels):
        cols = np.array([idwt_step(v[:r//2], v[r//2:], h) for v in A[:r, :c].T]).T
        A[:r, :c] = np.array([idwt_step(v[:c//2], v[c//2:], h) for v in cols])
        r *= 2; c *= 2
    return A

def keep_largest(Cc, frac):
    """Zero all but the largest frac of the coefficients (by magnitude)."""
    k = max(1, int(frac*Cc.size)); thr = np.sort(np.abs(Cc).ravel())[::-1][k - 1]
    return Cc*(np.abs(Cc) >= thr)

psnr = lambda a, b: 10*np.log10(1.0/np.mean((a - b)**2))     # peak signal-to-noise ratio for images scaled to [0, 1]

# . . The New York photograph of Module 6: gray, cropped to 512 x 1024, averaged in 2 x 2 blocks -> 256 x 512
ny = mpimg.imread('Fig/2.3_NEW_YORK.png')[:512, :1024, :3].mean(axis=2).astype(float)
img = ny.reshape(256, 2, 512, 2).mean(axis=(1, 3))
C_haar = dwt2(img, FILT['haar'], 5)

In Figure 13.12 the share of coefficients kept drops from 100% to 0.5%. Watch the reconstruction in panel (a): it barely changes
until only a few percent of the coefficients remain, and even at 0.5% the skyline is recognizable, in blocky form. Panel (b) shows why: the kept
coefficients (black) sit in the top-left corner, the coarse approximation, and in the detail blocks that record the edges of buildings.

In [ ]:
C = DSP_COLORS
keep_p = np.geomspace(1.0, 0.005, 20)                      # keep 100% ... 0.5% of the coefficients (20 frames)
n_frames = len(keep_p)
kept = [keep_largest(C_haar, p) for p in keep_p]
recs = [idwt2(Ck, FILT['haar'], 5) for Ck in kept]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(8.4, 2.6))
im_a = axa.imshow(recs[0], cmap='gray', vmin=0, vmax=1)
im_b = axb.imshow(kept[0] != 0, cmap='gray_r', vmin=0, vmax=1, interpolation='nearest')
axa.set_title('(a) Reconstruction', fontsize=11); axb.set_title('(b) Kept Haar coefficients (black)', fontsize=11)
for ax in (axa, axb):
    ax.set_xticks([]); ax.set_yticks([]); ax.axis('off')
title = fig.suptitle('', fontsize=12)

def update(k):
    im_a.set_data(recs[k]); im_b.set_data(kept[k] != 0)
    e_kept = np.sum(kept[k]**2)/np.sum(C_haar**2)
    p_txt = 'lossless' if k == 0 else f'PSNR {psnr(img, recs[k]):.1f} dB'
    title.set_text(f'keep {100*keep_p[k]:.2f}% of the coefficients:  {p_txt},  energy kept {100*e_kept:.2f}%')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=800, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 13.12.</b> Compressing an image with the 2D Haar transform (20 frames). (a) Reconstruction from the largest coefficients only; (b) where the kept coefficients sit in the 2D transform (black). Keeping 4.7% of the coefficients gives a PSNR of 22.7 dB with 85.08% of the energy; keeping 0.5% gives 19.0 dB. <i>(Python-generated.)</i>

How does the wavelet basis compare with the other bases of this course at equal storage? Figure 13.13 keeps 5% of the
131,072 numbers of the image in each of four ways.

In [ ]:
C = DSP_COLORS
frac = 0.05                                                 # keep 5% of the 256 x 512 numbers
k_store = int(frac*img.size)
rec_haar = idwt2(keep_largest(C_haar, frac), FILT['haar'], 5)
rec_db4 = idwt2(keep_largest(dwt2(img, FILT['db4'], 5), frac), FILT['db4'], 5)
F_img = np.fft.fft2(img)
rec_fft = np.real(np.fft.ifft2(keep_largest(F_img, frac)))  # conjugate pairs are kept together: k complex = k real numbers
U_i, S_i, Vt_i = np.linalg.svd(img, full_matrices=False)
k_svd = k_store//(img.shape[0] + img.shape[1] + 1)          # each rank-1 term costs m + n + 1 numbers (Module 8)
rec_svd = (U_i[:, :k_svd]*S_i[:k_svd]) @ Vt_i[:k_svd]
storage = {'Haar': rec_haar, 'Daubechies-4': rec_db4, '2D FFT': rec_fft, f'SVD, rank {k_svd}': rec_svd}
fig, axs = plt.subplots(2, 2, figsize=(8.8, 4.7))
for ax, (lab, r), let in zip(axs.flat, storage.items(), 'abcd'):
    ax.imshow(np.clip(r, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'({let}) {lab}: PSNR {psnr(img, r):.1f} dB', fontsize=12); ax.set_xticks([]); ax.set_yticks([]); ax.axis('off')
plt.tight_layout(); plt.show()

<b>Figure 13.13.</b> The same image stored in 5% of its numbers (6,553 values) four ways: (a) the largest Haar coefficients; (b) the largest Daubechies-4 coefficients; (c) the largest 2D FFT coefficients (conjugate pairs count as two real numbers); (d) a rank-8 SVD (Module 8), each rank-1 term costing $m + n + 1$ numbers. PSNR: 22.9, 23.1, 20.9 and 18.7 dB. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; another orthonormal basis.</b> An orthonormal DWT is a square matrix $\mathbf{W}$ with $\mathbf{W}^T\mathbf{W} = \mathbf{I}$, like the
$\mathbf{U}$ and $\mathbf{V}$ of the SVD in Module 8 (Section 8.6), so dropping coefficients removes exactly their energy (equation 15).
The SVD finds the best basis for this one image but needs many numbers per basis vector; the wavelet basis is fixed in advance and is sparse for
edges, which is why it wins at equal storage in Figure 13.13.
</div>

<a name="13.8"></a>
## 13.8 Parting Thoughts

Wavelets replace the single window of the STFT with a family of windows whose length scales with the
period being analyzed. The CWT is a redundant, picture-friendly transform, best for seeing how the frequency content of a geophysical record
changes; the DWT is an exact, orthonormal transform with $N$ coefficients, best for processing: denoising, compression and fast multiscale
analysis. Both are built from operations met earlier in the course: correlation (Module 7), the FFT (Module 11), downsampling (Module 10) and
orthonormal bases (Module 8). The filter pairs $h$ and $g$ of Section 13.6 are analyzed as Z-transform polynomials in Module 14.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 13.1 &mdash; Reading scales and scalograms</b>

The Ricker wavelet $\psi(t) = (1 - t^2/a^2)\,e^{-t^2/(2a^2)}$ has amplitude spectrum $(a\omega)^2 e^{-(a\omega)^2/2}$ (up to a constant). A Morlet CWT of the Tohoku record (Figure 13.7, computed at exactly 0.01 Hz and 0.05 Hz) has its largest values at 5240 s (0.01 Hz, a 100 s period) and 6083 s (0.05 Hz, 20 s) after 05:00:00 UTC. The earthquake origin time was 05:46:24 UTC (2784 s on this clock) and the epicentral distance to station WET is 81.8° = 9,098 km.

<b>(a)</b> Set the derivative of the spectrum to zero to show that its peak frequency is $f_p = \sqrt{2}/(2\pi a)$ (MATH 213). What scale $a$ gives a 25 Hz Ricker wavelet, and where are its zero crossings?

<b>(b)</b> Compute the group velocity $U$ = distance/(arrival time − origin time) at the 100 s and 20 s periods. Which period arrives first, and why does that agree with surface-wave dispersion in POTE1 (GPGN 328)?

<b>(c)</b> A classmate says "the CWT has better frequency resolution than the STFT at every frequency." Test the claim with the Morlet bandwidth $\sigma_f = 1/(2\sqrt{2}\pi a)$, $a = 6/(2\pi f)$ (equation 11), against the 60 s window (resolution about $1/60$ Hz) used for the Tohoku spectrogram in Module 12.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 13.1</b></summary>

<b>(a)</b> $\frac{d}{d\omega}\left[(a\omega)^2 e^{-(a\omega)^2/2}\right] = a\,(a\omega)\left[2 - (a\omega)^2\right]e^{-(a\omega)^2/2} = 0$, so $a\omega = \sqrt{2}$ and $f_p = \sqrt{2}/(2\pi a) = 0.2251/a$. For $f_p$ = 25 Hz, $a = 0.2251/25$ = 9.00 ms, and $\psi = 0$ where $t^2 = a^2$: at $t = \pm a = \pm$9.0 ms.

<b>(b)</b> 100 s: $U$ = 9,098/(5240 − 2784) = 9,098/2456 = 3.70 km/s. 20 s: $U$ = 9,098/(6083 − 2784) = 2.76 km/s. The 100 s energy arrives about 843 s earlier: longer periods sample deeper, faster parts of the mantle, so they travel faster (normal dispersion of surface waves).

<b>(c)</b> Since $a = 6/(2\pi f)$, $\sigma_f = f/(6\sqrt{2})$. At 1 Hz: $a$ = 0.955 s, $\sigma_f$ = 0.118 Hz ($\sigma_t$ = 0.68 s). At 0.05 Hz: $a$ = 19.1 s, $\sigma_f$ = 0.0059 Hz ($\sigma_t$ = 13.5 s). The STFT resolution is about $1/60$ = 0.017 Hz at every frequency, so the CWT is finer only below $f = 6\sqrt{2}/60$ = 0.14 Hz and coarser above it. The claim is false at high frequency, where the CWT trades frequency resolution for time resolution (equation 3).
</details>

<div class="alert alert-warning">
<b>Microquestion 13.2 &mdash; Haar by hand</b>

An 8-sample trace is $x = [4, 6, 10, 12, 8, 6, 5, 5]$. Use unnormalized Haar averages $a[n] = (x[2n] + x[2n+1])/2$ and differences $d[n] = (x[2n] - x[2n+1])/2$.

<b>(a)</b> Compute one level of averages and differences, then repeat on the averages until one average remains.

<b>(b)</b> Repeat level 1 with the orthonormal factor $1/\sqrt{2}$ of equation (12) and show that the energy $\sum x^2$ equals $\sum a^2 + \sum d^2$ (Parseval, MATH 348; Module 11).

<b>(c)</b> "Keeping only every second sample aliases (Module 10), so the Haar transform must lose information." Explain why perfect reconstruction is still possible.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 13.2</b></summary>

<b>(a)</b> Level 1: $a = [5, 11, 7, 5]$, $d = [-1, -1, 1, 0]$. Level 2: $a = [8, 6]$, $d = [-3, 1]$. Level 3: $a = [7]$, $d = [1]$. The final average, 7, is the mean of the trace.

<b>(b)</b> $a_1 = [7.071, 15.556, 9.900, 7.071]$, $d_1 = [-1.414, -1.414, 1.414, 0]$. $\sum x^2 = 16 + 36 + 100 + 144 + 64 + 36 + 25 + 25 = 446$; $\sum a^2 = 50 + 242 + 98 + 50 = 440$ and $\sum d^2 = 2 + 2 + 2 + 0 = 6$, total 446.

<b>(c)</b> Each channel alone is undersampled, but together they hold $N$ numbers for $N$ samples, and $x[2n] = a + d$, $x[2n+1] = a - d$ (unnormalized form) recover every sample. In the language of filters, the aliasing in the two channels cancels in the synthesis bank (Figure 13.8). Numerically, a six-level reconstruction of a random signal is exact to 2e-15.
</details>

<div class="alert alert-warning">
<b>Microquestion 13.3 &mdash; How much can we throw away?</b>

The orthonormal three-level Haar coefficients of $x = [2, 2, 2, 2, 6, 6, 6, 10]$ are $[a_3, d_3, d_2, d_2, d_1, d_1, d_1, d_1] = [12.728, -7.071, 0, -2.000, 0, 0, 0, -2.828]$.

<b>(a)</b> How many numbers are needed to store $x$ exactly, and what is the compression ratio? Using equation (15) (Parseval’s theorem, MATH 348), what fraction of the energy is lost if only the two largest coefficients are kept? The three largest?

<b>(b)</b> The same energy bookkeeping governs a rank-$k$ SVD (Module 8, $\sum\sigma^2$). In Figure 13.13 the Haar DWT, the 2D FFT and the SVD each kept 5% of the numbers of the New York image. Explain the ranking using what each basis is good at.

<b>(c)</b> Critique: "wavelets beat Fourier for every signal." For a 1024-sample sinusoid exactly on a DFT bin, how many FFT coefficients hold 99% of the energy, and how many Daubechies-4 coefficients? Explain.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 13.3</b></summary>

<b>(a)</b> Four nonzero coefficients replace eight samples: 8:4 = 2:1, with nothing lost. $\sum x^2 = 224$. Keeping the two largest (12.728 and −7.071) drops $2.000^2 + 2.828^2 = 4 + 8 = 12$, i.e. 12/224 = 5.4% of the energy; the reconstruction is $[2, 2, 2, 2, 7, 7, 7, 7]$. Keeping the three largest (adding −2.828) drops only $2.000^2 = 4$, i.e. 4/224 = 1.8% of the energy, with reconstruction $[2, 2, 2, 2, 7, 7, 5, 9]$.

<b>(b)</b> PSNR 22.9 dB (Haar) > 20.9 dB (FFT) > 18.7 dB (SVD). A photograph is mostly smooth patches separated by edges. Haar and Daubechies wavelets need only a few coefficients per edge, because each basis function is local. The FFT spreads every edge over many frequencies, so it needs more coefficients for the same error. A rank-$k$ SVD can only represent images that are sums of a few outer products, rows times columns; a city skyline is not, and each rank-1 term costs $m + n + 1$ numbers.

<b>(c)</b> The FFT needs 1 coefficient; Daubechies-4 needs 309. A stationary sinusoid is a single Fourier basis function, but it overlaps every wavelet at every scale. Wavelets win for localized features such as edges and transients; Fourier wins for stationary oscillations, the stationarity assumption of Section 13.1.
</details>